# pcg_hr: final results
Summaries of the final-candidate runs in `output/runs/` (test split, fast setting, synthetic stress), the PSoC 6 cost model and the report figures. Runs are produced with `pcg-hr eval`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from pcg_hr import config, cost_model, plots

cfg = config.load()
RUNS = {"default (3.0 s)": "*_final_test_final", "fast (2.5 s)": "*_fast_test_final",
        "stress": "*_final_test_final_stress"}
run_dirs = {k: sorted(cfg["paths"]["runs_dir"].glob(v))[-1] for k, v in RUNS.items()}

## Test results

In [ ]:
keys = ["n", "first_within", "first_within_hr_ge_low", "first_within_low_hr", "first_half_double",
        "steady_within", "steady_loa", "ttfr_median_hr_ge_low_s", "ttfr_p95_hr_ge_low_s",
        "ttfr_median_low_hr_s", "ttfr_max_s", "clean_window_first_within", "sep_n", "targets_passed"]
summaries = {k: json.loads((d / "summary.json").read_text()) for k, d in run_dirs.items()}
pd.DataFrame({k: {key: s.get(key) for key in keys} for k, s in summaries.items()}).round(3)

## First-reading accuracy per noise type and SNR (dB)

In [ ]:
stress = pd.read_csv(run_dirs["stress"] / "stress_by_condition.csv")
stress.pivot(index="noise", columns="snr_db", values="first_within").round(3)

## PSoC 6 cost model (estimate)

In [ ]:
table = cost_model.profile(cfg)
display(table.drop(columns=["ram_detail", "per_frame"]).round(0))
cost_model.totals(table, cfg)

## Figures

In [ ]:
fig_dir = Path("../hr_notes/figures/final")
for name in ("accuracy_vs_time", "ttfr_cdf", "band_accuracy"):
    if (fig_dir / f"{name}.png").is_file():
        display(Image(fig_dir / f"{name}.png"))